# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [1]:
# Dependencias
import re, math, tarfile, zipfile
import numpy as np
import pandas as pd
import zstandard
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

In [ ]:
# Recoleccion del corpus 
pd.set_option("display.max_colwidth", 80)

CORPUS_PATH = "txt-files.tar.zip"
N_DOCS      = 1000
MIN_CHARS   = 5_000

START_RE = re.compile(r"\*\*\* ?START OF (THE|THIS) PROJECT GUTENBERG.*?\*\*\*", re.I | re.S)
END_RE   = re.compile(r"\*\*\* ?END OF (THE|THIS) PROJECT GUTENBERG", re.I)
TITLE_RE = re.compile(r"^Title:\s*(.+)$", re.M)

def limpiar_gutenberg(raw):
    m = START_RE.search(raw)
    if m: raw = raw[m.end():]
    m = END_RE.search(raw)
    if m: raw = raw[:m.start()]
    return raw.strip()

def iterar_docs(path):
    if path.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            nombres = z.namelist()
            tars = [n for n in nombres if n.lower().endswith((".tar", ".tar.gz", ".tgz"))]
            if tars:
                with z.open(tars[0]) as inner:
                    modo = "r|gz" if tars[0].lower().endswith((".gz", ".tgz")) else "r|"
                    tar = tarfile.open(fileobj=inner, mode=modo)
                    for member in tar:
                        if member.isfile() and member.name.lower().endswith(".txt"):
                            f = tar.extractfile(member)
                            yield member.name, f.read().decode("utf-8", errors="ignore")
            else:
                for name in nombres:
                    if name.lower().endswith(".txt"):
                        yield name, z.read(name).decode("utf-8", errors="ignore")
    else:
        if path.endswith(".zst"):
            fobj = zstandard.ZstdDecompressor().stream_reader(open(path, "rb"))
            tar = tarfile.open(fileobj=fobj, mode="r|")
        elif path.endswith((".gz", ".tgz")):
            tar = tarfile.open(path, mode="r|gz")
        else:
            tar = tarfile.open(path, mode="r|")
        for member in tar:
            if member.isfile() and member.name.lower().endswith(".txt"):
                f = tar.extractfile(member)
                yield member.name, f.read().decode("utf-8", errors="ignore")


In [3]:
# Preprocesamiento 
TOKEN_RE = re.compile(r"[a-z]+")
STOP = set(ENGLISH_STOP_WORDS)

def tokenizar(text):
    tokens = TOKEN_RE.findall(text.lower())
    return [t for t in tokens if len(t) > 2 and t not in STOP]

In [4]:
# Generacion de la tabla
ids, tits, txts = [], [], []
for name, raw in iterar_docs(CORPUS_PATH):
    if "Language: English" not in raw[:5000]:
        continue
    t = TITLE_RE.search(raw[:5000])
    body = limpiar_gutenberg(raw)
    if len(body) < MIN_CHARS:
        continue
    ids.append(f"D{len(ids):04d}")
    tits.append(t.group(1).strip() if t else name)
    txts.append(body)
    if len(ids) >= N_DOCS:
        break

corpus = pd.DataFrame({"doc_id": ids, "titulo": tits, "texto": txts})
corpus["tokens"] = corpus["texto"].apply(tokenizar)

doc_ids  = corpus["doc_id"].tolist()
titulos  = corpus["titulo"].tolist()
textos   = corpus["texto"].tolist()
docs     = corpus["tokens"].tolist()
N        = len(corpus)
longitudes = np.array([len(d) for d in docs])      
avgdl      = longitudes.mean()  

In [5]:
# Calculo de TF e IDF
def calcular_tf(docs):
    tfs = []
    for tokens in docs:
        tf = {}
        for t in tokens:
            tf[t] = tf.get(t, 0) + 1
        tfs.append(tf)
    return tfs

def calcular_df(tfs):
    DF = {}
    for tf in tfs:
        for t in tf:
            DF[t] = DF.get(t, 0) + 1
    return DF

tfs = calcular_tf(docs)
DF  = calcular_df(tfs)

# IDF
idf_tfidf = {t: math.log(N / d) + 1.0 for t, d in DF.items()}
# Para el bm25
idf_bm25  = {t: math.log(1 + (N - d + 0.5) / (d + 0.5)) for t, d in DF.items()}

In [6]:
print(f"Documentos           : {N}")
print(f"Vocabulario          : {len(DF):,} términos")
print(f"Longitud |d| (tokens): mín={longitudes.min():,} | media (avgdl)={avgdl:,.1f} | máx={longitudes.max():,}")
display(corpus[["doc_id", "titulo"]].assign(n_tokens=longitudes).head(10))


orden_df = sorted(DF.items(), key=lambda x: -x[1])
ejemplos = orden_df[:5] + orden_df[-5:]
display(pd.DataFrame({
    "Término": [t for t, _ in ejemplos],
    "DF": [d for _, d in ejemplos],
    "IDF (TF-IDF)": [round(idf_tfidf[t], 4) for t, _ in ejemplos],
    "IDF (BM25)": [round(idf_bm25[t], 4) for t, _ in ejemplos],
}))

Documentos           : 1000
Vocabulario          : 339,287 términos
Longitud |d| (tokens): mín=442 | media (avgdl)=29,517.1 | máx=452,822


,doc_id,titulo,n_tokens
0,D0000,The Magna Carta,5751
1,D0001,Apocolocyntosis,2323
2,D0002,The House on the Borderland,20756
3,D0003,"My First Years as a Frenchwoman, 1876-1879",26577
4,D0004,The Warriors,21982
5,D0005,A Voyage to the Moon,28184
6,D0006,La Fiammetta,5362
7,D0007,Carmilla,11362
8,D0008,The Mystery,29626
9,D0009,"Wild Northern Scenes; Or, Sporting Adventures with the Rifle and the Rod",36148


,Término,DF,IDF (TF-IDF),IDF (BM25)
0,little,998,1.0020,0.0025
1,come,997,1.0030,0.0035
2,time,996,1.0040,0.0045
3,like,995,1.0050,0.0055
4,long,993,1.0070,0.0075
5,gentryville,1,7.9078,6.5033
6,offut,1,7.9078,6.5033
7,outwrestle,1,7.9078,6.5033
8,pappsville,1,7.9078,6.5033
9,skivery,1,7.9078,6.5033


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [7]:
# Consultas
UMBRAL_ALTO, UMBRAL_BAJO = 0.30, 0.03 # Para categorizar las frecuencias de documentos que contienen los terminos  

def categoria_df(d):
    p = d / N
    return "alta" if p >= UMBRAL_ALTO else ("baja" if p < UMBRAL_BAJO else "media")

consultas = {
    "Q1": {"texto": "king",
           "necesidad": "Reyes y monarquía: reinados, cortes y sucesiones (1 término, se espera df alta)."},
    "Q2": {"texto": "harpoon whale whaling",
           "necesidad": "Relatos sobre la caza de ballenas (3 términos, se espera df baja)."},
    "Q3": {"texto": "garden flowers roses spring",
           "necesidad": "Jardinería y flores según la estación (4 términos, df mixta)."},
    "Q4": {"texto": "physician fever disease treatment",
           "necesidad": "Medicina: enfermedades, fiebres y su tratamiento (4 términos, df mixta)."},
    "Q5": {"texto": "opera composer symphony violin concert",
           "necesidad": "Música clásica: compositores, óperas y conciertos (5 términos, df media-baja)."},
    "Q6": {"texto": "philosophy reason nature knowledge truth",
           "necesidad": "Ensayos filosóficos sobre razón y conocimiento (5 términos, df alta-media)."},
    "Q7": {"texto": "law court judge trial",
           "necesidad": "Derecho y procesos judiciales (4 términos, df media-alta)."},
    "Q8": {"texto": "a journey across the mountains by train to reach the distant city in winter",
           "necesidad": "Relatos de viaje largo en tren hacia una ciudad lejana en invierno (consulta larga, 7 tokens)."},
}

filas_df = []
for qid, q in consultas.items():
    q["tokens"] = tokenizar(q["texto"])
    q["terminos"] = [t for t in dict.fromkeys(q["tokens"]) if t in DF]   
    print(f"\n{qid}: «{q['texto']}»")
    print(f"   Necesidad: {q['necesidad']}")
    print(f"   Tokens   : {q['tokens']}")
    for t in dict.fromkeys(q["tokens"]):
        d = DF.get(t, 0)
        print(f"   {t:<12} df={d:>4}  ({d / N:6.1%})  -> {categoria_df(d) if d else 'ausente del vocabulario'}")
    dfs = [DF[t] for t in q["terminos"]]
    filas_df.append({"Consulta": qid, "Tokens": len(q["tokens"]), "En vocabulario": len(q["terminos"]),
                     "df mín": min(dfs, default=0), "df máx": max(dfs, default=0)})

display(pd.DataFrame(filas_df))

def terminos_con_df(minimo, maximo, n=15):
    cand = [(t, d) for t, d in DF.items() if minimo <= d <= maximo and len(t) > 3]
    return sorted(cand, key=lambda x: -x[1])[:n]
# print(terminos_con_df(5, 30))


Q1: «king»
   Necesidad: Reyes y monarquía: reinados, cortes y sucesiones (1 término, se espera df alta).
   Tokens   : ['king']
   king         df= 780  ( 78.0%)  -> alta

Q2: «harpoon whale whaling»
   Necesidad: Relatos sobre la caza de ballenas (3 términos, se espera df baja).
   Tokens   : ['harpoon', 'whale', 'whaling']
   harpoon      df=  19  (  1.9%)  -> baja
   whale        df= 149  ( 14.9%)  -> media
   whaling      df=  29  (  2.9%)  -> baja

Q3: «garden flowers roses spring»
   Necesidad: Jardinería y flores según la estación (4 términos, df mixta).
   Tokens   : ['garden', 'flowers', 'roses', 'spring']
   garden       df= 698  ( 69.8%)  -> alta
   flowers      df= 695  ( 69.5%)  -> alta
   roses        df= 434  ( 43.4%)  -> alta
   spring       df= 842  ( 84.2%)  -> alta

Q4: «physician fever disease treatment»
   Necesidad: Medicina: enfermedades, fiebres y su tratamiento (4 términos, df mixta).
   Tokens   : ['physician', 'fever', 'disease', 'treatment']
   physician  

,Consulta,Tokens,En vocabulario,df mín,df máx
0,Q1,1,1,780,780
1,Q2,3,3,19,149
2,Q3,4,4,434,842
3,Q4,4,4,382,557
4,Q5,5,5,65,288
5,Q6,5,5,451,912
6,Q7,4,4,612,840
7,Q8,7,7,585,835


In [8]:
# TF-IDF 
vectorizer = TfidfVectorizer(analyzer=lambda tokens: tokens, use_idf=True,
                             smooth_idf=False, sublinear_tf=False, norm=None)
X_tfidf = vectorizer.fit_transform(docs).tocsr()
norma_docs = np.sqrt(np.asarray(X_tfidf.multiply(X_tfidf).sum(axis=1)).ravel())   

def similitud_coseno(tokens):
    q = vectorizer.transform([tokens])                  
    if q.nnz == 0:
        return np.zeros(N)
    norma_q = np.sqrt(q.multiply(q).sum())
    productos = (X_tfidf @ q.T).toarray().ravel()       
    sim = np.zeros(N)
    np.divide(productos, norma_q * norma_docs, out=sim, where=norma_docs > 0)
    return sim

In [9]:
# Ajuste de k1 y b
K1, B = 1.2, 0.75

# BM25 
def aporte_bm25(t, i, k1=K1, b=B):
    f = tfs[i].get(t, 0)
    if f == 0 or t not in idf_bm25:
        return 0.0
    K = k1 * (1 - b + b * longitudes[i] / avgdl)
    return idf_bm25[t] * f * (k1 + 1) / (f + K)

def scores_bm25(tokens, k1=K1, b=B):
    terminos = [t for t in dict.fromkeys(tokens) if t in idf_bm25]    # términos únicos en el vocabulario
    return np.array([sum(aporte_bm25(t, i, k1, b) for t in terminos) for i in range(N)])

In [10]:
# Ordenamiento de score descedentemente
def ordenar(scores, k=10):
    positivos = [i for i in range(N) if scores[i] > 0]
    positivos.sort(key=lambda i: (-scores[i], doc_ids[i]))
    return positivos if k is None else positivos[:k]

for qid, q in consultas.items():
    q["sim_tfidf"] = similitud_coseno(q["tokens"])
    q["sc_bm25"]   = scores_bm25(q["tokens"])
    q["top_tfidf"] = ordenar(q["sim_tfidf"], 10)
    q["top_bm25"]  = ordenar(q["sc_bm25"], 10)

prueba = tokenizar("zzzqqq xxyyzz")
assert ordenar(similitud_coseno(prueba)) == [] and ordenar(scores_bm25(prueba)) == []

In [11]:
def tabla_ranking(lista, scores, nombre):
    return pd.DataFrame({
        "Posición": range(1, len(lista) + 1),
        "ID": [doc_ids[i] for i in lista],
        "Título": [titulos[i][:50] for i in lista],
        nombre: [round(scores[i], 4) for i in lista],
    })

for qid, q in consultas.items():
    print("=" * 90)
    print(f"{qid}: «{q['texto']}»")
    print(f"\n--- TF-IDF + coseno ({len(q['top_tfidf'])} documentos) ---")
    display(tabla_ranking(q["top_tfidf"], q["sim_tfidf"], "Coseno")) if q["top_tfidf"] else print("(lista vacía)")
    print(f"--- BM25 k1={K1}, b={B} ({len(q['top_bm25'])} documentos) ---")
    display(tabla_ranking(q["top_bm25"], q["sc_bm25"], "Score BM25")) if q["top_bm25"] else print("(lista vacía)")

Q1: «king»

--- TF-IDF + coseno (10 documentos) ---


,Posición,ID,Título,Coseno
0,1,D0362,Henry the Second,0.4149
1,2,D0202,The Cavalier Songs and Ballads of England from 164,0.3427
2,3,D0344,Arthurian Chronicles: Roman de Brut,0.2977
3,4,D0422,"The History of England, Volume I",0.2824
4,5,D0961,Comic History of England,0.2615
5,6,D0857,Grimm's Fairy Stories,0.2506
6,7,D0482,Coming to the King,0.2486
7,8,D0534,"A Political Diary, 1828-1830, Volume II",0.2484
8,9,D0404,"The Life of Marie Antoinette, Queen of France",0.2459
9,10,D0155,"The Great Events by Famous Historians, Volume 05",0.2416


--- BM25 k1=1.2, b=0.75 (10 documentos) ---


,Posición,ID,Título,Score BM25
0,1,D0362,Henry the Second,0.5463
1,2,D0585,The Story of the Champions of the Round Table,0.5462
2,3,D0173,The Everlasting Whisper,0.5461
3,4,D0344,Arthurian Chronicles: Roman de Brut,0.5461
4,5,D0534,"A Political Diary, 1828-1830, Volume II",0.5460
5,6,D0422,"The History of England, Volume I",0.5459
6,7,D0202,The Cavalier Songs and Ballads of England from 164,0.5459
7,8,D0860,"King Henry VI, Part 3",0.5458
8,9,D0516,William the Conqueror,0.5458
9,10,D0404,"The Life of Marie Antoinette, Queen of France",0.5457


Q2: «harpoon whale whaling»

--- TF-IDF + coseno (10 documentos) ---


,Posición,ID,Título,Coseno
0,1,D0928,"Jack Mason, the Old Sailor",0.1916
1,2,D0469,Within the Deep,0.0767
2,3,D0015,"Punchinello, Volume 1, No. 19, August 6, 1870",0.0536
3,4,D0394,"The Sea Lions; Or, The Lost Sealers",0.0301
4,5,D0302,Trips to the Moon,0.0295
5,6,D0449,Lost in the Air,0.0249
6,7,D0077,"A Narrative of the Life and Adventures of Venture,",0.0245
7,8,D0599,Thrilling Adventures by Land and Sea,0.0209
8,9,D0043,Aleutian Indian and English Dictionary,0.0207
9,10,D0125,Fairies and Fusiliers,0.0180


... (22 outputs left out)

Q8: «a journey across the mountains by train to reach the distant city in winter»

--- TF-IDF + coseno (10 documentos) ---


,Posición,ID,Título,Coseno
0,1,D0753,The Lands of the Saracen,0.1167
1,2,D0452,A General History and Collection of Voyages and Tr,0.1142
2,3,D0687,My Native Land,0.0924
3,4,D0316,American Big Game in Its Haunts: The Book of the B,0.0897
4,5,D0489,The Youthful Wanderer,0.0846
5,6,D0843,"Letters of a Traveller; Or, Notes of Things Seen i",0.0814
6,7,D0768,After Waterloo: Reminiscences of European Travel 1,0.0797
7,8,D0866,A woman's journey round the world,0.0777
8,9,D0590,"Oregon, Washington and Alaska; Sights and Scenes f",0.0748
9,10,D0157,The Oregon Trail: Sketches of Prairie and Rocky-Mo,0.0644


--- BM25 k1=1.2, b=0.75 (10 documentos) ---


,Posición,ID,Título,Score BM25
0,1,D0032,"The Life of Hon. William F. Cody, Known as Buffalo",4.5552
1,2,D0969,The Expedition of the Donner Party and its Tragic,4.5548
2,3,D0968,The Discovery of Yellowstone Park,4.5336
3,4,D0687,My Native Land,4.5046
4,5,D0843,"Letters of a Traveller; Or, Notes of Things Seen i",4.4991
5,6,D0285,Around the World in Eighty Days,4.4895
6,7,D0157,The Oregon Trail: Sketches of Prairie and Rocky-Mo,4.4749
7,8,D0503,"""De Bello Gallico"" and Other Commentaries",4.4388
8,9,D0112,With British Guns in Italy: A Tribute to Italian A,4.4291
9,10,D0452,A General History and Collection of Voyages and Tr,4.4144


**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [12]:
COLS = ["Posición", "ID y título TF-IDF", "Coseno", "ID y título BM25", "Score BM25"]

def tabla_comparacion(qid):
    q = consultas[qid]
    tt, tb = q["top_tfidf"], q["top_bm25"]
    filas = []
    for r in range(max(len(tt), len(tb))):
        a = tt[r] if r < len(tt) else None
        b_ = tb[r] if r < len(tb) else None
        filas.append({
            "Posición": r + 1,
            "ID y título TF-IDF": f"{doc_ids[a]} · {titulos[a][:40]}" if a is not None else "",
            "Coseno": round(q["sim_tfidf"][a], 4) if a is not None else "",
            "ID y título BM25": f"{doc_ids[b_]} · {titulos[b_][:40]}" if b_ is not None else "",
            "Score BM25": round(q["sc_bm25"][b_], 4) if b_ is not None else "",
        })
    return pd.DataFrame(filas, columns=COLS)

resumen = []
for qid, q in consultas.items():
    tt, tb = q["top_tfidf"], q["top_bm25"]
    ids_t, ids_b = {doc_ids[i] for i in tt}, {doc_ids[i] for i in tb}
    comunes = ids_t & ids_b
    misma_pos = sum(1 for r in range(min(len(tt), len(tb))) if tt[r] == tb[r])
    print("=" * 90)
    print(f"{qid}: «{q['texto']}»")
    display(tabla_comparacion(qid))
    resumen.append({
        "Consulta": qid,
        "|T_TFIDF|": len(tt), "|T_BM25|": len(tb),
        "|T_TFIDF ∩ T_BM25|": len(comunes),
        "Misma posición": misma_pos,
        "1.º TF-IDF": doc_ids[tt[0]] if tt else "—",
        "1.º BM25": doc_ids[tb[0]] if tb else "—",
        "¿Mismo 1.º?": bool(tt and tb and tt[0] == tb[0]),
    })

print("\nComparación")
display(pd.DataFrame(resumen))

Q1: «king»


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,D0362 · Henry the Second,0.4149,D0362 · Henry the Second,0.5463
1,2,D0202 · The Cavalier Songs and Ballads of Englan,0.3427,D0585 · The Story of the Champions of the Round,0.5462
2,3,D0344 · Arthurian Chronicles: Roman de Brut,0.2977,D0173 · The Everlasting Whisper,0.5461
3,4,"D0422 · The History of England, Volume I",0.2824,D0344 · Arthurian Chronicles: Roman de Brut,0.5461
4,5,D0961 · Comic History of England,0.2615,"D0534 · A Political Diary, 1828-1830, Volume II",0.5460
5,6,D0857 · Grimm's Fairy Stories,0.2506,"D0422 · The History of England, Volume I",0.5459
6,7,D0482 · Coming to the King,0.2486,D0202 · The Cavalier Songs and Ballads of Englan,0.5459
7,8,"D0534 · A Political Diary, 1828-1830, Volume II",0.2484,"D0860 · King Henry VI, Part 3",0.5458
8,9,"D0404 · The Life of Marie Antoinette, Queen of F",0.2459,D0516 · William the Conqueror,0.5458
9,10,"D0155 · The Great Events by Famous Historians, V",0.2416,"D0404 · The Life of Marie Antoinette, Queen of F",0.5457


Q2: «harpoon whale whaling»


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,"D0928 · Jack Mason, the Old Sailor",0.1916,"D0394 · The Sea Lions; Or, The Lost Sealers",18.3121
1,2,D0469 · Within the Deep,0.0767,D0449 · Lost in the Air,16.4955
2,3,"D0015 · Punchinello, Volume 1, No. 19, August 6,",0.0536,D0229 · An Antarctic Mystery,16.4809
3,4,"D0394 · The Sea Lions; Or, The Lost Sealers",0.0301,"D0015 · Punchinello, Volume 1, No. 19, August 6,",15.0300
4,5,D0302 · Trips to the Moon,0.0295,"D0677 · Natalie; Or, A Gem Among the Sea-Weeds",14.1558
5,6,D0449 · Lost in the Air,0.0249,"D0928 · Jack Mason, the Old Sailor",12.0691
6,7,D0077 · A Narrative of the Life and Adventures o,0.0245,"D0181 · Kalitan, Our Little Alaskan Cousin",11.4862
7,8,D0599 · Thrilling Adventures by Land and Sea,0.0209,D0469 · Within the Deep,10.7767
8,9,D0043 · Aleutian Indian and English Dictionary,0.0207,D0043 · Aleutian Indian and English Dictionary,10.1861
9,10,D0125 · Fairies and Fusiliers,0.0180,D0077 · A Narrative of the Life and Adventures o,9.3674


Q3: «garden flowers roses spring»


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,D0682 · Hardy Ornamental Flowering Trees and Shr,0.1809,D0461 · Grass of Parnassus,3.6933
1,2,D0461 · Grass of Parnassus,0.1288,D0121 · The Folk-lore of Plants,3.6717
2,3,D0121 · The Folk-lore of Plants,0.1110,D0175 · Poems,3.6160
3,4,"D0518 · The Botanic Garden. Part 2, Containing t",0.0850,D0913 · Poems,3.6065
4,5,"D0589 · The Miracle, and Other Poems",0.0826,"D0589 · The Miracle, and Other Poems",3.5944
5,6,D0329 · The Lonely Dancer and Other Poems,0.0821,D0033 · The Complete Poetical Works of Edgar All,3.5809
6,7,D0175 · Poems,0.0808,D0595 · A Florida Sketch-Book,3.5533
7,8,"D0970 · Phebe, the Blackberry Girl",0.0712,"D0428 · Poems, with The Ballad of Reading Gaol",3.5445
8,9,"D0741 · Robert Louis Stevenson, an Elegy; and Ot",0.0637,D0446 · Cap and Gown,3.5404
9,10,"D0829 · Renascence, and Other Poems",0.0633,D0844 · Christmas in Legend and Story: A Book fo,3.5279


... (8 outputs left out)

Q8: «a journey across the mountains by train to reach the distant city in winter»


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,D0753 · The Lands of the Saracen,0.1167,"D0032 · The Life of Hon. William F. Cody, Known",4.5552
1,2,D0452 · A General History and Collection of Voya,0.1142,D0969 · The Expedition of the Donner Party and i,4.5548
2,3,D0687 · My Native Land,0.0924,D0968 · The Discovery of Yellowstone Park,4.5336
3,4,D0316 · American Big Game in Its Haunts: The Boo,0.0897,D0687 · My Native Land,4.5046
4,5,D0489 · The Youthful Wanderer,0.0846,"D0843 · Letters of a Traveller; Or, Notes of Thi",4.4991
5,6,"D0843 · Letters of a Traveller; Or, Notes of Thi",0.0814,D0285 · Around the World in Eighty Days,4.4895
6,7,D0768 · After Waterloo: Reminiscences of Europea,0.0797,D0157 · The Oregon Trail: Sketches of Prairie an,4.4749
7,8,D0866 · A woman's journey round the world,0.0777,"D0503 · ""De Bello Gallico"" and Other Commentarie",4.4388
8,9,"D0590 · Oregon, Washington and Alaska; Sights an",0.0748,D0112 · With British Guns in Italy: A Tribute to,4.4291
9,10,D0157 · The Oregon Trail: Sketches of Prairie an,0.0644,D0452 · A General History and Collection of Voya,4.4144



Comparación


,Consulta,|T_TFIDF|,|T_BM25|,|T_TFIDF ∩ T_BM25|,Misma posición,1.º TF-IDF,1.º BM25,¿Mismo 1.º?
0,Q1,10,10,6,1,D0362,D0362,True
1,Q2,10,10,7,1,D0928,D0394,False
2,Q3,10,10,4,1,D0682,D0461,False
3,Q4,10,10,7,1,D0269,D0269,True
4,Q5,10,10,5,0,D0786,D0499,False
5,Q6,10,10,5,1,D0923,D0923,True
6,Q7,10,10,4,1,D0465,D0575,False
7,Q8,10,10,4,0,D0753,D0032,False


In [13]:
QID_ANALISIS = None     # Para poder elegir cualquier consulta automaticamente

def rangos(lista):
    return {i: r + 1 for r, i in enumerate(lista)}

def mejor_par(qid):
    """Par de documentos comunes a ambos top 10 con mayor cambio de posición relativa (prioriza inversiones)."""
    q = consultas[qid]
    rt, rb = rangos(q["top_tfidf"]), rangos(q["top_bm25"])
    comunes = [i for i in rt if i in rb]
    mejor = None
    for x in range(len(comunes)):
        for y in range(x + 1, len(comunes)):
            a, b_ = comunes[x], comunes[y]
            dt, db = rt[b_] - rt[a], rb[b_] - rb[a]
            clave = (dt * db < 0, abs(dt - db))
            if mejor is None or clave > mejor[0]:
                mejor = (clave, a, b_)
    return mejor

if QID_ANALISIS is None:
    cands = {qid: mejor_par(qid) for qid in consultas}
    cands = {k: v for k, v in cands.items() if v}
    QID_ANALISIS = max(cands, key=lambda k: cands[k][0])
(_, d1, d2) = mejor_par(QID_ANALISIS)
q = consultas[QID_ANALISIS]
print(f"Consulta analizada: {QID_ANALISIS} «{q['texto']}»  | términos en vocabulario: {q['terminos']}")

def detalle(qid, i, k1=K1, b=B):
    q = consultas[qid]
    qvec = vectorizer.transform([q["tokens"]])
    norma_q = np.sqrt(qvec.multiply(qvec).sum())
    K = k1 * (1 - b + b * longitudes[i] / avgdl)
    filas = []
    for t in q["terminos"]:
        j = vectorizer.vocabulary_[t]
        f = tfs[i].get(t, 0)
        q_i, d_i = qvec[0, j], X_tfidf[i, j]
        filas.append({
            "Término": t,
            "|d|": longitudes[i],
            "|d|/avgdl": longitudes[i] / avgdl,
            "f(t,d)": f,
            "df(t)": DF[t],
            "IDF BM25": idf_bm25[t],
            "K": K,
            "Saturación f(k1+1)/(f+K)": f * (k1 + 1) / (f + K),
            "Aporte BM25": aporte_bm25(t, i, k1, b),
            "q_i·d_i": q_i * d_i,
            "Aporte coseno": q_i * d_i / (norma_q * norma_docs[i]),
        })
    return pd.DataFrame(filas)

rt, rb = rangos(q["top_tfidf"]), rangos(q["top_bm25"])
for i in (d1, d2):
    det = detalle(QID_ANALISIS, i)
    assert np.isclose(det["Aporte BM25"].sum(), q["sc_bm25"][i])           # los aportes suman el score
    assert np.isclose(det["Aporte coseno"].sum(), q["sim_tfidf"][i])
    print("\n" + "-" * 90)
    print(f"{doc_ids[i]} · {titulos[i]}")
    print(f"|d| = {longitudes[i]:,} (avgdl = {avgdl:,.1f}, razón {longitudes[i] / avgdl:.2f}) | ||d|| TF-IDF = {norma_docs[i]:.2f}")
    print(f"TF-IDF: posición {rt[i]}, coseno = {q['sim_tfidf'][i]:.4f}  |  BM25: posición {rb[i]}, score = {q['sc_bm25'][i]:.4f}")
    display(det.round(4))

Consulta analizada: Q4 «physician fever disease treatment»  | términos en vocabulario: ['physician', 'fever', 'disease', 'treatment']

------------------------------------------------------------------------------------------
D0091 · Thaumaturgia; Or, Elucidations of the Marvellous
|d| = 40,191 (avgdl = 29,517.1, razón 1.36) | ||d|| TF-IDF = 1436.61
TF-IDF: posición 2, coseno = 0.0547  |  BM25: posición 10, score = 5.3093


,Término,|d|,|d|/avgdl,"f(t,d)",df(t),IDF BM25,K,Saturación f(k1+1)/(f+K),Aporte BM25,q_i·d_i,Aporte coseno
0,physician,40191,1.3616,40,382,0.9620,1.5255,2.1192,2.0387,154.0303,0.0313
1,fever,40191,1.3616,10,532,0.6312,1.5255,1.9088,1.2048,26.6053,0.0054
2,disease,40191,1.3616,31,529,0.6368,1.5255,2.0968,1.3353,83.0492,0.0169
3,treatment,40191,1.3616,2,557,0.5853,1.5255,1.2481,0.7305,5.0257,0.0010



------------------------------------------------------------------------------------------
D0325 · A Practical Physiology: A Text-Book for Higher Schools
|d| = 66,834 (avgdl = 29,517.1, razón 2.26) | ||d|| TF-IDF = 3941.53
TF-IDF: posición 10, coseno = 0.0360  |  BM25: posición 3, score = 5.7330


,Término,|d|,|d|/avgdl,"f(t,d)",df(t),IDF BM25,K,Saturación f(k1+1)/(f+K),Aporte BM25,q_i·d_i,Aporte coseno
0,physician,66834,2.2642,21,382,0.9620,2.3378,1.9796,1.9044,80.8659,0.0060
1,fever,66834,2.2642,24,532,0.6312,2.3378,2.0047,1.2653,63.8526,0.0047
2,disease,66834,2.2642,99,529,0.6368,2.3378,2.1492,1.3687,265.2216,0.0197
3,treatment,66834,2.2642,30,557,0.5853,2.3378,2.0410,1.1946,75.3848,0.0056


## **Explicación**
Se eligieron los documentos D0091 (Thaumaturgia; Or, Elucidations of the Marvellous) y D0325 (A Practical Physiology); que tienen una mayor longitud comparada al promedio del corpus; para la consulta «physician fever disease treatment». TF-IDF pone a D0091 en el puesto 2 (coseno 0.0547) y a D0325 en el 10 (0.0360). BM25 los invierte: D0325 queda tercero (5.7330) y D0091 décimo (5.3093). 

En TF-IDF todo se explica por la norma del documento. D0325 tiene un producto punto mayor (485.3 frente a 268.7), porque con peso lineal en la frecuencia disease aparece 99 veces. Pero su norma es 3941.5 frente a 1436.6, es decir 2.74 veces mayor. El producto punto crece 1.81 veces y la norma 2.74, así que su coseno queda en 0.66 veces el de D0091.

BM25 no cae en eso porque satura la frecuencia: physician aparece 40 veces en D0091 y 21 en D0325, pero aporta 2.0387 frente a 1.9044, apenas 0.13 de diferencia. Lo que decide es qué documento cubre mejor los cuatro términos. treatment aparece solo 2 veces en D0091 y 30 en D0325 . Esa diferencia de +0.4641 supera por sí sola la ventaja total de D0325 (+0.4237), mientras que fever y disease suman +0.0939 y physician resta 0.1343. La longitud pesa poco: D0325 es más largo (K = 2.34 frente a 1.53), pero con frecuencias altas casi no se penaliza. Tampoco hay un término raro que domine, porque los cuatro tienen df entre 38 % y 56 % (IDF entre 0.59 y 0.96).

**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [15]:
# Prueba de ajustes de parámetros
CONFIGS = {
    "Base":              (1.2, 0.75),
    "Sin ajuste (b=0)":  (1.2, 0.00),
    "Mayor k1 (2.0)":    (2.0, 0.75),
}
NOMBRES = list(CONFIGS)
QID_PARAMS = None        # Nuevamnente para poder elegir automaticamente una consulta

# Elegir la consulta cuyo top-10 BM25 tenga la mayor variación de longitud entre documentos
def variacion_longitud(qid):
    top = consultas[qid]["top_bm25"]
    if len(top) < 5:
        return -1
    l = longitudes[top]
    return l.std() / l.mean()

if QID_PARAMS is None:
    QID_PARAMS = max(consultas, key=variacion_longitud)
q = consultas[QID_PARAMS]
print(f"Consulta: {QID_PARAMS} «{q['texto']}»  | términos: {q['terminos']}")
print(f"Variación de longitud en su top-10 BM25 (desv/media): {variacion_longitud(QID_PARAMS):.2f}")

corridas = {}
for nombre, (k1, b) in CONFIGS.items():
    sc = scores_bm25(q["tokens"], k1, b)
    orden = ordenar(sc, None)
    corridas[nombre] = {"scores": sc, "orden": orden, "pos": rangos(orden)}

union = []
for nombre in NOMBRES:
    for i in corridas[nombre]["orden"][:10]:
        if i not in union:
            union.append(i)
union.sort(key=lambda i: corridas["Base"]["pos"][i])

filas = []
for i in union:
    fila = {"ID": doc_ids[i], "Título": titulos[i][:35],
            "|d|": longitudes[i], "|d|/avgdl": round(longitudes[i] / avgdl, 2)}
    for t in q["terminos"]:
        fila[f"f({t})"] = tfs[i].get(t, 0)
    for nombre in NOMBRES:
        fila[f"Pos {nombre}"] = corridas[nombre]["pos"][i]
    for nombre in NOMBRES:
        fila[f"Score {nombre}"] = round(corridas[nombre]["scores"][i], 4)
    filas.append(fila)
tabla_pos = pd.DataFrame(filas)
display(tabla_pos)

def se_mueve(i):
    return len({corridas[n]["pos"][i] for n in NOMBRES}) > 1

movidos  = [i for i in union if se_mueve(i)]
estables = [i for i in union if not se_mueve(i)]
print(f"Documentos que cambian de posición: {len(movidos)} de {len(union)}")

Consulta: Q2 «harpoon whale whaling»  | términos: ['harpoon', 'whale', 'whaling']
Variación de longitud en su top-10 BM25 (desv/media): 1.11


,ID,Título,|d|,|d|/avgdl,f(harpoon),f(whale),f(whaling),Pos Base,Pos Sin ajuste (b=0),Pos Mayor k1 (2.0),Score Base,Score Sin ajuste (b=0),Score Mayor k1 (2.0)
0,D0394,"The Sea Lions; Or, The Lost Sealers",68585,2.32,14,84,18,1,1,1,18.3121,19.3736,23.2994
1,D0449,Lost in the Air,19146,0.65,4,22,2,2,3,2,16.4955,15.4777,20.0704
2,D0229,An Antarctic Mystery,33155,1.12,3,7,11,3,2,3,16.4809,16.7508,20.0047
3,D0015,"Punchinello, Volume 1, No. 19, Augu",6265,0.21,1,12,1,4,5,4,15.0300,11.2656,17.6523
4,D0677,"Natalie; Or, A Gem Among the Sea-We",26874,0.91,5,5,1,5,4,5,14.1558,13.8853,16.4482
5,D0928,"Jack Mason, the Old Sailor",1847,0.06,4,21,0,6,6,6,12.0691,10.6220,15.8352
6,D0181,"Kalitan, Our Little Alaskan Cousin",10862,0.37,4,14,0,7,7,7,11.4862,10.5178,14.6605
7,D0469,Within the Deep,6480,0.22,0,36,3,8,8,8,10.7767,9.5866,13.8592
8,D0043,Aleutian Indian and English Diction,1518,0.05,1,3,0,9,17,9,10.1861,6.9263,12.2754
9,D0077,A Narrative of the Life and Adventu,3842,0.13,0,1,2,10,21,10,9.3674,6.7475,11.2107


Documentos que cambian de posición: 8 de 12


In [16]:
def tabla_aportes(docs_sel):
    filas = []
    for i in docs_sel:
        for t in q["terminos"]:
            fila = {"ID": doc_ids[i], "Término": t, "f(t,d)": tfs[i].get(t, 0),
                    "|d|/avgdl": round(longitudes[i] / avgdl, 2), "IDF BM25": round(idf_bm25[t], 3)}
            for nombre, (k1, b) in CONFIGS.items():
                fila[f"Aporte {nombre}"] = round(aporte_bm25(t, i, k1, b), 4)
            filas.append(fila)
    return pd.DataFrame(filas)

if movidos:
    print("\nAportes por término: documentos que SE MUEVEN")
    display(tabla_aportes(movidos[:3]))
if estables:
    print("Aportes por término: documentos que NO se mueven")
    display(tabla_aportes(estables[:3]))


Aportes por término: documentos que SE MUEVEN


,ID,Término,"f(t,d)",|d|/avgdl,IDF BM25,Aporte Base,Aporte Sin ajuste (b=0),Aporte Mayor k1 (2.0)
0,D0449,harpoon,4,0.65,3.938,7.0964,6.6649,8.6352
1,D0449,whale,22,0.65,1.901,4.0217,3.9668,5.3464
2,D0449,whaling,2,0.65,3.524,5.3774,4.8460,6.0888
3,D0229,harpoon,3,1.12,3.938,6.0296,6.1888,6.8363
4,D0229,whale,7,1.12,1.901,3.5234,3.5710,4.3474
5,D0229,whaling,11,1.12,3.524,6.9280,6.9910,8.8210
6,D0015,harpoon,1,0.21,3.938,5.8110,3.9383,6.4976
7,D0015,whale,12,0.21,1.901,4.0188,3.8029,5.3402
8,D0015,whaling,1,0.21,3.524,5.2002,3.5244,5.8146


Aportes por término: documentos que NO se mueven


,ID,Término,"f(t,d)",|d|/avgdl,IDF BM25,Aporte Base,Aporte Sin ajuste (b=0),Aporte Mayor k1 (2.0)
0,D0394,harpoon,14,2.32,3.938,7.4004,7.9803,9.1969
1,D0394,whale,84,2.32,1.901,4.0674,4.1243,5.4460
2,D0394,whaling,18,2.32,3.524,6.8444,7.2690,8.6565
3,D0928,harpoon,4,0.06,3.938,7.9557,6.6649,10.2877
4,D0928,whale,21,0.06,1.901,4.1134,3.9571,5.5475
5,D0928,whaling,0,0.06,3.524,0.0000,0.0000,0.0000
6,D0181,harpoon,4,0.37,3.938,7.4835,6.6649,9.3548
7,D0181,whale,14,0.37,1.901,4.0027,3.8530,5.3057
8,D0181,whaling,0,0.37,3.524,0.0000,0.0000,0.0000


## **Explicación**
Se usó la consulta «harpoon whale whaling», con 12 documentos entre los tres top 10.

Con b = 0  cambian 8 de los 12 documentos, favoreciendo a los largos y perjudicando a los cortos. A Narrative of the Life and Adventures of Venture (que es más corto que la media) baja del puesto 10 al 21, y Aleutian Indian and English Dictionary (igualmente corto) del 9 al 17. Punchinello solo baja del 4 al 5 y su score cae de 15.0300 a 11.2656, porque sus términos aparecen una sola vez y con b = 0 el aporte de harpoon pasa de 5.8110 a 3.9383. En cambio, The Return of Sherlock Holmes sube del 11 al 9 y Thrilling Adventures by Land and Sea del 12 al 10. An Antarctic Mystery pasa del 3 al 2 y rebasa a Lost in the Air, cuyos tres aportes bajan.

Con k1 = 2 . El tope de cada aporte sube por tres y los scores aumentan entre 16 % y 31 %, pero ningún documento cambia de puesto. El aumento depende de la frecuencia. En Punchinello, harpoon sube 11.8 %, mientras que whale (con 12 apariciones) sube 32.9 %. Aun así las distancias no se invierten.

En resumen, b controla cuánto pesa la longitud y es el que reordena el ranking. k1 controla cuánto vale repetir un término y sobre todo cambia la escala de los scores.